# 05 — PPO Training (Corrected)
Uses the repository PPO implementation. No in-notebook source patching. Rewards come from explicit benchmark execution.

In [ ]:
!pip install -q 'trl<0.12.0' peft transformers datasets bitsandbytes accelerate
import os,sys,shutil,torch
repo=os.path.abspath(os.getcwd())
if os.path.isdir('/kaggle/input') and not os.path.exists(os.path.join(repo,'src')):
    for root,dirs,files in os.walk('/kaggle/input'):
        if 'src' in dirs and os.path.exists(os.path.join(root,'src','training','ppo.py')):
            shutil.copytree(os.path.join(root,'src'),os.path.join('/kaggle/working','src'),dirs_exist_ok=True); repo='/kaggle/working'; break
sys.path.insert(0,repo)
from datasets import load_dataset
from transformers import AutoTokenizer
from src.training.ppo import run_ppo_training
MODEL='deepseek-ai/deepseek-coder-1.3b-instruct'
SFT='./checkpoints/sft/final'
if not os.path.isfile(os.path.join(SFT,'adapter_config.json')):
    raise FileNotFoundError('SFT adapter checkpoint required: '+SFT)
if not os.path.isfile(os.path.join(SFT,'adapter_model.safetensors')):
    raise FileNotFoundError('SFT adapter weights missing: '+SFT)
print('Verified SFT adapter:', SFT)
apps=load_dataset('codeparrot/apps',revision='refs/convert/parquet',split='train[:16]')
apps=apps.filter(lambda x: bool(x.get('solutions')))
tok=AutoTokenizer.from_pretrained(MODEL,trust_remote_code=True)
print('PPO smoke dataset:',len(apps))

In [ ]:
from src.execution.executor import PythonSandbox
from src.rewards.execution_reward import compute_partial_reward
sandbox=PythonSandbox(default_timeout=5.0,max_memory_mb=1024.0)
tests=[{"assertion":"assert add(1,2) == 3"},{"assertion":"assert add(2,3) == 5"}]
cases={
    "perfect":"def add(a,b): return a+b",
    "partial":"def add(a,b): return a+b if a==1 else 0",
    "bad":"def add(a,b): return a+",
}
for name,code in cases.items():
    result=sandbox.run_tests(code,tests)
    reward=compute_partial_reward(result)
    print(f"{name}: status={result.status}, passed={result.passed_tests}/{result.total_tests}, reward={reward:.3f}")
assert compute_partial_reward(sandbox.run_tests(cases["perfect"],tests)) == 1.0
partial_reward=compute_partial_reward(sandbox.run_tests(cases["partial"],tests))
assert 0.0 < partial_reward < 1.0
assert compute_partial_reward(sandbox.run_tests(cases["bad"],tests)) < 0.0
print("PASS: execution reward plumbing produces perfect, partial, and negative rewards.")

In [ ]:
trainer=run_ppo_training(SFT,tok,apps,output_dir='./checkpoints/ppo_smoke',batch_size=1,mini_batch_size=1,gradient_accumulation_steps=1,max_steps=2)
print('PPO smoke test finished. Inspect reward variation and checkpoint before scaling up.')

### Scaling rule
Only after the 2-step smoke test succeeds with non-empty executable tests and a valid SFT adapter should the dataset/steps be increased. The old 10-step result is retained as historical evidence, not as the final PPO result.